# Advanced Training in Neural‑LAM

This tutorial covers advanced training topics including loss functions, hyperparameter tuning, multi-GPU training, and experiment tracking.

## 1. Understanding Loss Functions

Neural-LAM implements several loss functions for different forecasting scenarios:

### Deterministic Losses

- **`wmse`** (Weighted MSE) - Default. Balances variables by inverse variance
- **`mse`** - Standard Mean Squared Error
- **`wmae`** (Weighted MAE) - Robust to outliers, balanced variables
- **`mae`** - Standard Mean Absolute Error

### Probabilistic Losses

- **`nll`** - Negative Log Likelihood for uncertainty estimation
- **`crps_gauss`** - Continuous Ranked Probability Score (proper scoring rule)

**Note:** Probabilistic losses require `--output_std` flag to enable uncertainty estimation.

## 2. Training with Different Loss Functions

### Example 1: Weighted MSE (Default)

```bash
python -m neural_lam.train_model \
  --config_path data/config.yaml \
  --model graph_lam \
  --graph flat_graph \
  --loss wmse \
  --epochs 20
```

### Example 2: Mean Absolute Error

```bash
python -m neural_lam.train_model \
  --config_path data/config.yaml \
  --model graph_lam \
  --graph flat_graph \
  --loss mae \
  --epochs 20
```

### Example 3: Probabilistic Forecasting

```bash
python -m neural_lam.train_model \
  --config_path data/config.yaml \
  --model graph_lam \
  --graph flat_graph \
  --loss nll \
  --output_std \
  --epochs 20
```

## 3. Hyperparameter Tuning

Key hyperparameters to experiment with:

### Model Architecture

```bash
# Small model (faster training)
python -m neural_lam.train_model \
  --config_path data/config.yaml \
  --model graph_lam \
  --graph flat_graph \
  --hidden_dim 32 \
  --hidden_layers 1 \
  --processor_layers 2

# Large model (more capacity)
python -m neural_lam.train_model \
  --config_path data/config.yaml \
  --model graph_lam \
  --graph flat_graph \
  --hidden_dim 128 \
  --hidden_layers 3 \
  --processor_layers 6 \
  --lr 5e-4
```

### Key Parameters

| Parameter | Description | Typical Values |
|-----------|-------------|----------------|
| `--hidden_dim` | Size of hidden representations | 32, 64, 128, 256 |
| `--hidden_layers` | Depth of encoder/decoder MLPs | 1-4 |
| `--processor_layers` | Number of message-passing steps | 2-8 |
| `--lr` | Learning rate | 1e-4 to 1e-3 |
| `--ar_steps_train` | Autoregressive rollout length | 1-10 |
| `--batch_size` | Batch size | 2-16 |

## 4. Multi-GPU Training

### Single Node, Multiple GPUs

```bash
# Use GPUs 0 and 1
python -m neural_lam.train_model \
  --config_path data/config.yaml \
  --model graph_lam \
  --graph flat_graph \
  --devices 0 1
```

### Multi-Node (HPC Cluster)

```bash
# 4 nodes with 4 GPUs each
python -m neural_lam.train_model \
  --config_path data/config.yaml \
  --model graph_lam \
  --graph flat_graph \
  --num_nodes 4 \
  --devices 4
```

## 5. Mixed Precision Training

Speed up training with reduced precision:

```bash
# 16-bit mixed precision
python -m neural_lam.train_model \
  --config_path data/config.yaml \
  --model graph_lam \
  --graph flat_graph \
  --precision 16

# Brain Float 16 (recommended for A100+ GPUs)
python -m neural_lam.train_model \
  --config_path data/config.yaml \
  --model graph_lam \
  --graph flat_graph \
  --precision bf16
```

## 6. Resuming and Fine-Tuning

### Resume Training (with optimizer state)

```bash
python -m neural_lam.train_model \
  --config_path data/config.yaml \
  --model graph_lam \
  --graph flat_graph \
  --load saved_models/my_model/last.ckpt \
  --restore_opt
```

### Fine-Tune (reset optimizer)

```bash
python -m neural_lam.train_model \
  --config_path data/config.yaml \
  --model graph_lam \
  --graph flat_graph \
  --load saved_models/my_model/best.ckpt
```

## 7. Loss Weighting Configuration

Control how much each atmospheric variable contributes to the loss.

### Uniform Weighting (Default)

Edit `data/config.yaml`:

```yaml
datastore:
  kind: mdp
  config_path: danra.datastore.yaml

training:
  state_feature_weighting:
    __config_class__: UniformFeatureWeighting
```

### Manual Weighting

```yaml
training:
  state_feature_weighting:
    __config_class__: ManualStateFeatureWeighting
    weights:
      u100m: 1.0    # Wind u-component
      v100m: 1.0    # Wind v-component
      r2m: 1.5      # Relative humidity (higher weight)
      t2m: 2.0      # Temperature (highest weight)
```

Higher weights mean that variable contributes more to the loss.

## 8. Experiment Tracking with Weights & Biases

Neural-LAM has built-in W&B support:

```bash
# First time: authenticate
wandb login

# Train with W&B logging (default)
python -m neural_lam.train_model \
  --config_path data/config.yaml \
  --model graph_lam \
  --graph flat_graph \
  --logger wandb \
  --logger-project my_weather_project
```

W&B automatically logs:
- Training and validation losses
- Learning rate schedules
- Model architecture
- System metrics (GPU usage, etc.)
- Example predictions

## 9. Complete Training Example

Here's a complete command with commonly used options:

```bash
python -m neural_lam.train_model \
  --config_path data/config.yaml \
  --model graph_lam \
  --graph flat_graph \
  --loss wmse \
  --hidden_dim 64 \
  --hidden_layers 2 \
  --processor_layers 4 \
  --lr 1e-3 \
  --batch_size 4 \
  --epochs 50 \
  --ar_steps_train 3 \
  --ar_steps_eval 10 \
  --precision 16 \
  --logger wandb \
  --logger-project neural_lam
```

This trains a GraphLAM model with:
- Weighted MSE loss
- 64-dimensional hidden representations
- 4 message-passing layers
- 3-step rollout during training
- 10-step rollout during validation
- Mixed precision (16-bit)
- W&B experiment tracking